# 02_local_inference: Run a Model Yourself

[Open in Colab](https://colab.research.google.com/github/Utkarsh-09/AI_GURU_labs/blob/main/notebooks/02_local_inference.ipynb)

**Session:** Day 2, S7 — Lab: run a model yourself (50-minute block)
**Expected runtime:** **40 minutes** of participant time: two TODOs, reading what the server sends back, and one side-by-side comparison. Machine time is small and reported in two parts, because they have different causes. **Pull** (downloading the model, once per machine): `llama3.2:1b` is 1.3 GB; **28 s** on a Colab T4 runtime over Google's network, minutes on venue Wi-Fi. **Run** (everything after the pull): about **one minute** on the build laptop with the model already pulled, and about a second a reply on a T4. On Colab, installing Ollama itself adds about **56 s**.
**Needs:** Google Colab (free tier, **T4 GPU runtime** - *Runtime > Change runtime type*; a CPU runtime is not refused, but it has not been measured on Colab and two container tests on the build machine ranged from 30 s to many minutes per reply, so take the T4) — or a laptop with Ollama installed (`setup/ollama_setup.md`). **No API key for the local part.** The single comparison call at the end needs `OPENAI_API_KEY` (Colab Secrets, or `.env` locally); without it, everything up to that cell still runs. Data from the repo: `data/eval/heldout_20.jsonl`.
**A correct result looks like:** the final cell prints `LOCAL INFERENCE READY` with the Ollama version, the model card (`1.2B` parameters, `Q8_0`), a tokens-per-second figure, and the same synthetic ticket answered by the self-hosted model and by the vendor API, side by side, with a verdict on each reply's format and fields. The small local model gets several **fields** wrong, and its **format** is not reliable either: the same request, at temperature 0, came back as valid JSON in one sitting on the build laptop and without its closing brace an hour later on the same laptop and in a Linux container. That is expected — S10 to S12 are about fixing it.

> All data in this lab is synthetic. No real OQ material anywhere.

---

## Self-hosted versus vendor API — not laptop versus cloud

Yesterday every model call left the building: your ticket text went to a vendor's servers and an answer came back. Today the model runs on a machine **you** control, and nothing leaves it. The question this lab sets up for the rest of Day 2 is not *"is a small local model as good as GPT?"* (it is not) but **"what does it take to own the model, and what do you get for it?"**

The Colab runtime you are about to use is a stand-in for **OQ's own Azure VM** — the deployment your S4 cost model priced. Same shape: a Linux box with one GPU, a model server on it, an HTTP API on a port, and your integration talking to that port. The absolute speeds here (a T4, or two CPU cores) are not the VM's speeds; the *behaviour* — pull once, load once, one request at a time, tokens per second — is identical, and it is what the sizing worksheet in S8 works from. If you run this on your laptop instead, everything below is the same except the numbers.

**The model:** `llama3.2:1b`, the same model Day 2 fine-tunes this afternoon, so one pull serves notebooks 02, 03 and 06. It is small on purpose — 1.3 GB downloads on venue Wi-Fi, and answers in a second on a T4. `llama3.2:3b` answers noticeably better and is one string away in the settings cell; a production VM would run 8B or larger. Size is a dial, not a verdict.

**The model pull is the live risk of this session, and it is separable.** The cell marked **PULL CELL** downloads the model, does nothing if it is already there, and can be run on its own the day before — that is what the Day 1 tech check is for, on laptops. On Colab the runtime is thrown away when you disconnect, so the pull happens again on every fresh runtime; it is fast there (28 s measured) because Colab's network is not venue Wi-Fi.

---
**The plan.** Get an Ollama server answering (install it if this is Colab) → **pull** the model, or skip if pulled → look at what was downloaded → call it with **raw HTTP** and read every field of the reply → the same call through `config/endpoints.py` → the server's own clock: tokens per second → **TODO 1:** generation parameters — same prompt, three settings, twice each → a real ticket through the local model with the house system prompt → **TODO 2:** the same ticket through the vendor API → one table.

**If the runtime disconnects:** reconnect and *Run all*. Every result is saved to your checkpoint folder as it is produced, and cells that already have their result load it instead of asking again. On a fresh Colab runtime the Ollama install and the pull run again; nothing else does.

**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [1]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

Environment : local
Repo root   : C:\Users\utkar\oq-advanced-ai
Checkpoints : C:\Users\utkar\oq-advanced-ai\checkpoints\local


**Why this cell:** each notebook installs only what it needs, with exact pins matching `requirements.txt`. This one needs `requests` alone — Colab preinstalls it at exactly this version, so on Colab the line is a no-op that costs a second. **Ollama is not a Python package.** It is a server program, and it gets its own cell below.

In [2]:
# Pinned installs — versions match requirements.txt. Colab only;
# local machines installed requirements.txt during setup.
if IN_COLAB:
    %pip install -q requests==2.32.4
print("Install cell done.")

Install cell done.


**Why this cell:** only the **last** part of this lab talks to the vendor API, and only to put one answer next to the local model's. The key must never be typed into a cell (notebooks get committed). Locally it lives in the repo-root `.env`; on Colab it comes from **Colab Secrets** (key icon in the left sidebar, a secret named exactly `OPENAI_API_KEY`, *Notebook access* on) or a one-time hidden paste. Unlike notebook 01, a missing key does **not** stop this notebook here: the self-hosted part is the lab, so it runs regardless, and only the comparison cell near the end will stop and say why.

In [3]:
import utils  # shared helpers from notebooks/utils.py

key_ok = utils.ensure_api_key(IN_COLAB)
if not key_ok:
    print("No hosted key. Everything up to the comparison cell (TODO 2) runs without it.")

OPENAI_API_KEY: found in .env


**Why this cell:** the choices that shape the lab sit in one place. `MODEL_NAME` is the model you will pull and run — the 1B by default (see the header for why). The line that sets `OLLAMA_MODEL` is how `config/endpoints.py` learns what the word `local` means in this notebook: it reads that variable, and nothing else in the notebook mentions the model name to the endpoint. The server writes its log to a git-ignored folder in the repo; it is the first place to look when a model will not load.

In [4]:
import json
import time

import inference_utils
import ollama_utils
import utils
from config.endpoints import get_endpoint

MODEL_NAME = "llama3.2:1b"    # the model this lab runs; "llama3.2:3b" answers better and is one edit away

# config/endpoints.py reads OLLAMA_MODEL to know which model "local" means. Tell it.
os.environ["OLLAMA_MODEL"] = MODEL_NAME

server_log_dir = REPO_ROOT / "eval_runs"      # `ollama serve` writes its log here (git-ignored)
sitting_started = time.time()

print(f"model to run : {MODEL_NAME}")
print(f"server       : {ollama_utils.server_url()}   (OLLAMA_BASE_URL; the same address the `ollama` command uses)")
print(f"checkpoints  : {CHECKPOINT_DIR}")

model to run : llama3.2:1b
server       : http://localhost:11435   (OLLAMA_BASE_URL; the same address the `ollama` command uses)
checkpoints  : C:\Users\utkar\oq-advanced-ai\checkpoints\local


**Why this cell — the server:** a model server is a program that holds model weights in memory and answers HTTP requests on a port. On OQ's VM it would be installed once and left running. Here the helper does the same job in one call: if a server already answers on `OLLAMA_BASE_URL`, it is used as it is (your laptop's Ollama, for instance); if Ollama is installed but not running, it starts `ollama serve` in the background and waits until it answers; on Colab, where nothing is installed, it first downloads **one pinned release** (0.12.10, the release every number in this repo was measured on — never "latest") and unpacks it. Each of those steps is a command you could type in a terminal; the guide `setup/ollama_setup.md` lists them.

It also checks for a GPU first. On Colab without one the lab still works, but every reply takes many seconds instead of one — the cell says so rather than leaving you to wonder.

In [5]:
gpu = ollama_utils.gpu_name()
print(f"NVIDIA GPU: {gpu or 'none found'}")
if IN_COLAB and gpu is None:
    print("This Colab runtime has no GPU. The lab is not refused here, but it has NOT been measured on a")
    print("Colab CPU runtime: two-core containers on the build machine gave 3 tokens a second (a 30 s ticket")
    print("reply) in one shape and minutes per reply in another. Runtime > Change runtime type > T4 GPU")
    print("(free tier), then Run all - nothing is lost. Stay here only if the GPU quota is gone.")

server = ollama_utils.ensure_server(IN_COLAB, log_dir=server_log_dir)

print(f"Ollama {server['version']} answering at {server['url']}")
if server["installed_here"]:
    print(f"  installed by this cell: the pinned release {ollama_utils.OLLAMA_VERSION}, unpacked under /usr/local")
print(f"  {'started by this cell (`ollama serve`, in the background)' if server['started_here'] else 'was already running - used as it is'}")
print(f"  models already on this server: {ollama_utils.list_models() or 'none yet'}")

NVIDIA GPU: none found


Ollama 0.12.10 answering at http://localhost:11435
  was already running - used as it is


  models already on this server: ['gpt-oss-safeguard:20b', 'llama3.2:1b', 'llama3.2:3b', 'oq-ticket-tuned:latest', 'qwen3-coder:30b']


**Why this cell — PULL CELL, milestone 1:** `ollama pull` downloads the model's weights once and stores them on the machine (`~/.ollama/models` on a laptop, `/usr/share/ollama/.ollama/models` on Linux, `/root/.ollama/models` in Colab). The helper first asks the server what it already has and **skips the download if the model is there** — so this cell is safe to run twice, and safe to run on its own the day before. It prints progress every 10 percent; a pull with no output for a minute is a network problem, not a slow one.

The download time is saved to your checkpoint folder **separately** from everything else, because it has a different cause (the network) from every other number in this lab (the machine), and the timing log keeps them apart.

In [6]:
# PULL CELL. Separable: run this cell alone the day before. Afterwards it is a no-op.
pull_started = time.time()
pull_status = ollama_utils.ensure_model(MODEL_NAME)
pull_seconds = round(time.time() - pull_started, 1)

previous_pull = utils.load_json(CHECKPOINT_DIR, "02_pull", default=None)
if pull_status == "pulled" or previous_pull is None or previous_pull.get("model") != MODEL_NAME:
    pull_record = {"model": MODEL_NAME, "status": pull_status, "seconds": pull_seconds,
                   "where": "Colab" if IN_COLAB else "local", "when": time.strftime("%Y-%m-%dT%H:%M:%S")}
    utils.save_json(CHECKPOINT_DIR, "02_pull", pull_record)
else:
    pull_record = previous_pull
    print(f"(the download itself was recorded on {previous_pull['when']}: {previous_pull['seconds']} s)")

print(f"{MODEL_NAME}: {pull_status}  -  this cell took {pull_seconds} s")

checkpoint saved: C:\Users\utkar\oq-advanced-ai\checkpoints\local\02_pull.json
llama3.2:1b: already there  -  this cell took 2.1 s


**Why this cell — what did you download?** A model name hides three facts you need for sizing: the parameter count (`1.2B`), the **quantization** (`Q8_0`: each weight stored in 8 bits, which is why 1.2 billion weights fit in 1.3 GB), and the context length — and there are *two* of those. The model was trained to hold 131,072 tokens; **this server gives it 4,096**, because the memory for context grows with its length and a ticket plus the system prompt is under 1,000 tokens. That served figure is `OLLAMA_CONTEXT_LENGTH`, set when the server started. On a laptop where the desktop app was already running, it is whatever the app is set to — and if it is set very high, a 1.3 GB model can refuse to load for lack of memory (playbook E2).

The warm-up request loads the weights into memory. Read its two numbers: how long the load took (the first real request would otherwise pay it), and what share of the model sits in **GPU** memory — `100%` on a T4, `0%` on a plain CPU.

In [7]:
warm = ollama_utils.warm_up(MODEL_NAME)
card = inference_utils.model_card(MODEL_NAME)

inference_utils.print_model_card(card, served_context_length=warm["context_length"])
print()
print(f"warm-up: loaded and answered in {warm['seconds']} s; {warm['gpu_share']:.0%} of the model is in GPU memory")

model            : llama3.2:1b
family           : llama
parameters       : 1.2B
quantization     : Q8_0   (bits per weight in the download)
on disk          : 1.32 GB
context, maximum : 131072 tokens  (what the model was trained to hold)
context, served  : 4096 tokens  (what THIS server gives it: OLLAMA_CONTEXT_LENGTH)
capabilities     : completion, tools

warm-up: loaded and answered in 6.1 s; 0% of the model is in GPU memory


**Why this cell — the raw call:** this is the whole protocol. One `POST` to `/v1/chat/completions` with a model name and a list of messages; one JSON object back. It is the **same request shape the vendor API takes** — Ollama implements the OpenAI-compatible endpoint on purpose — which is why an integration written against one can be pointed at the other by changing a URL. Read every field of the reply, not just the text: `model` (what actually answered), `choices[0].finish_reason` (`stop` = the model chose to end; `length` = you cut it off), and `usage` (tokens in, tokens out — the unit everything is priced and sized in).

In [8]:
import requests

question = [{"role": "user", "content": "In one sentence, what does an IT service desk do?"}]

request_body = {"model": MODEL_NAME, "messages": question, "temperature": 0.0, "max_tokens": 80}
response = requests.post(f"{server['url']}/v1/chat/completions", json=request_body, timeout=600)

print(f"HTTP {response.status_code} from {response.url}")
print(json.dumps(response.json(), indent=2))

HTTP 200 from http://localhost:11435/v1/chat/completions
{
  "id": "chatcmpl-235",
  "object": "chat.completion",
  "created": 1790085071,
  "model": "llama3.2:1b",
  "system_fingerprint": "fp_ollama",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "An IT service desk is a centralized point of contact for customers to report and request technical issues or support with computer systems, networks, and other technology services, providing prompt resolution and assistance."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 37,
    "completion_tokens": 38,
    "total_tokens": 75
  }
}


**Why this cell — the same call through the endpoint module:** `config/endpoints.py` (Interface Contract #3) wraps exactly the request you just made — nothing more — and gives every lab one line to call a model. The switch is the string: `"local"` is this server, `"hosted"` is the vendor, `"tuned"` is this afternoon's fine-tune. Print the endpoint object and check its `base_url`: **localhost**. Nothing you send through it leaves this machine.

In [9]:
local_llm = get_endpoint("local")
print(local_llm)
print()

reply_text = local_llm.chat(messages=question, temperature=0.0, max_tokens=80)
print(reply_text)

Endpoint(name='local', model='llama3.2:1b', base_url='http://localhost:11435/v1')



An IT service desk is a centralized point of contact for customers to report and request technical issues or support with computer systems, networks, and other technology services, providing prompt resolution and assistance.


**Why this cell — milestone 2, the server's own clock:** the OpenAI-compatible reply tells you *how many* tokens; only Ollama's native `/api/chat` tells you *how long each stage took*, from the server's own clock, in nanoseconds. Three stages matter: **load** (bringing weights into memory — zero once loaded, and `OLLAMA_KEEP_ALIVE` decides how long they stay), **prompt** (reading your input, fast and parallel), and **generate** (writing the reply, one token at a time). Reply tokens divided by generate seconds is **tokens per second, for one request at a time** — the number the sizing worksheet in S8 starts from, and the number that collapses when several requests arrive at once (notebook 03 does exactly that to this server). A T4 gives roughly 100 tokens a second on this model; two CPU cores, a few.

In [10]:
speed = utils.load_json(CHECKPOINT_DIR, "02_speed", default=None)
if speed is None:
    speed = inference_utils.chat_timed(MODEL_NAME, question, options={"temperature": 0.0, "num_predict": 80})
    speed["model"] = MODEL_NAME
    speed["measured_on"] = gpu or ("Colab CPU" if IN_COLAB else "local CPU")
    utils.save_json(CHECKPOINT_DIR, "02_speed", speed)
else:
    print(f"(from checkpoint - measured earlier on: {speed['measured_on']})")

inference_utils.print_timing(speed)

checkpoint saved: C:\Users\utkar\oq-advanced-ai\checkpoints\local\02_speed.json
load model       :    0.11 s   (0 once it is already in memory)
read the prompt  :    0.07 s   (37 tokens)
write the reply  :    1.02 s   (38 tokens)
total            :    1.24 s
speed            :    37.3 tokens per second, one request at a time


**Why this cell — TODO 1, generation parameters:** the same prompt does not have to give the same answer. Four request fields decide how the model picks each next token: `temperature` (0 = always the likeliest token; 1 = sample in proportion; higher = flatter), `top_p` (only sample among the tokens that make up this much probability), `seed` (start the sampler from a fixed point), and `max_tokens` (a hard cap on the reply — the reply then ends with `finish_reason: "length"`, mid-sentence). An integration that parses replies wants the first kind of setting; one that drafts text for a human may want the second; every integration needs the cap, because a model that does not stop is a bill and a timeout.

Fill in three settings. Each is sent **twice** and the cell says whether the two replies came back identical. Before you run it, write down what you expect for each — then check. One honest caveat, measured in S12's build: temperature 0 makes a small model *mostly* repeatable, not perfectly; between sittings and machines, 9 to 12 of 20 replies from this model changed. Repeatable-by-setting is not the same as deterministic.

In [11]:
experiment_prompt = [{"role": "user", "content":
    "Write a two-sentence status update to a user whose laptop LAP-04412 is being replaced tomorrow."}]

# ── TODO 1 ─────────────────────────────────────────────────────────
# Three settings for the same prompt. Each is sent twice.
# Hint: the fields Ollama's OpenAI-compatible endpoint accepts are temperature (0.0 to 2.0),
# top_p (0.0 to 1.0), seed (an integer) and max_tokens (a hard cap on the reply).
#   "repeatable": settings that should give the SAME reply both times
#   "creative":   settings that should give a DIFFERENT reply each time
#   "capped":     temperature 0 plus a cap small enough to cut the reply off mid-sentence
experiments = {
    "repeatable": {"temperature": 0.0, "seed": 42},
    "creative":   {"temperature": 1.0},
    "capped":     {"temperature": 0.0, "max_tokens": 12},
}
# ───────────────────────────────────────────────────────────────────

unfilled = [name for name, settings in experiments.items() if any(value is ... for value in settings.values())]
assert unfilled == [], f"TODO 1 is not filled in yet: {unfilled}"

experiment_results = inference_utils.run_experiments(MODEL_NAME, experiment_prompt, experiments, repeats=2)
inference_utils.print_experiments(experiment_results)
saved_path = utils.save_json(CHECKPOINT_DIR, "02_parameter_runs", experiment_results)

=== repeatable: {"temperature": 0.0, "seed": 42}
  try 1: "I'm so sorry to inform you that your laptop, LAP-04412, will be going in for a replace...
         finish_reason=stop  tokens=44  3.56 s
  try 2: "I'm so sorry to inform you that your laptop, LAP-04412, will be going in for a replace...
         finish_reason=stop  tokens=44  3.53 s
  the 2 replies are IDENTICAL

=== creative: {"temperature": 1.0}
  try 1: "I wanted to let you know that your laptop, LAP-04412, will be being replaced tomorrow ...
         finish_reason=stop  tokens=55  3.84 s
  try 2: "I'm so sorry, but tomorrow I'll be on an unexpected trip and won't be able to make it ...
         finish_reason=stop  tokens=50  3.7 s
  the 2 replies are DIFFERENT (identical for the first 2 characters)

=== capped: {"temperature": 0.0, "max_tokens": 12}
  try 1: "I'm so sorry to inform you that your laptop, LAP
         finish_reason=length  tokens=12  2.53 s
  try 2: "I'm so sorry to inform you that your laptop, LAP
         f

**Why this cell — milestone 3, a real ticket:** now the model does the job the rest of Day 2 is about. The messages are taken straight from a held-out evaluation row: the **house system prompt** (`dataset_utils.SYSTEM_PROMPT`, the only copy in the repo) and one synthetic ticket; the row's expected answer stays out of the prompt and is used only to mark the reply. Two things to read in the output. **Format:** did the model return one JSON object and nothing else, as the prompt demands? A small model manages that most of the time — and not every time: in several sittings on the build machines this very request came back missing its closing brace, so a parser would have rejected it. The verdict you see below is whatever happened on your machine just now. **Fields:** how many of the six exact-match fields are right? A small untuned model usually gets several wrong — routing queues it has never heard of, an urgency level the prompt never defined. Hold on to both facts: the fine-tune in S11 exists to move the second one.

In [12]:
import dataset_utils

heldout_pairs = dataset_utils.load_jsonl(REPO_ROOT / "data" / "eval" / "heldout_20.jsonl")
ticket_pair = heldout_pairs[0]                          # INC-004183; change the index to try another
ticket_messages = ticket_pair["messages"][:2]           # [house system prompt, the ticket]; the answer stays hidden
expected_record = json.loads(ticket_pair["messages"][2]["content"])
exact_fields = {field: value for field, value in expected_record.items() if field != "requested_action"}

print(f"--- {ticket_pair['ticket_id']}: what both models are sent ---")
print(dataset_utils.pair_user_text(ticket_pair))
print()

local_result = utils.load_json(CHECKPOINT_DIR, "02_ticket_local", default=None)
if local_result is None or local_result["model"] != MODEL_NAME:
    call_started = time.time()
    local_reply = local_llm.chat(messages=ticket_messages, temperature=0.0, max_tokens=300, timeout=600)
    local_result = {"ticket_id": ticket_pair["ticket_id"], "model": MODEL_NAME, "endpoint": repr(local_llm),
                    "reply": local_reply, "seconds": round(time.time() - call_started, 2)}
    utils.save_json(CHECKPOINT_DIR, "02_ticket_local", local_result)

local_record, local_format = inference_utils.parse_json_reply(local_result["reply"])
local_marks = inference_utils.compare_records(local_record, exact_fields)

print(f"--- {MODEL_NAME} (self-hosted) replied in {local_result['seconds']} s ---")
print(local_result["reply"])
print()
print(f"format : {local_format}")
print(f"fields : {inference_utils.marks_text(local_marks, len(exact_fields))}")
print(f"expected: {json.dumps(expected_record)}")

--- INC-004183: what both models are sent ---
Subject: no mic in TEAMS

Greetings,

In TEAMS the other side is not hearing my voice. Mic is OK in other application. Today is the last date for month end closing, please support. Second thing, My password of Claimpoint is expired. Kindly reset the same. Requesting your kind support.

With regards,
Reem
Sent from my mobile



checkpoint saved: C:\Users\utkar\oq-advanced-ai\checkpoints\local\02_ticket_local.json
--- llama3.2:1b (self-hosted) replied in 5.17 s ---
{
  "category": "access",
  "affected_system": null,
  "asset_tag": null,
  "urgency": "medium",
  "impact": "single_user",
  "requested_action": "Activate mic in TEAMS and reset password for Claimpoint",
  "routing_queue": "identity_access"

format : not JSON
fields : no readable record (0/6)
expected: {"category": "software", "affected_system": "Teams", "asset_tag": null, "urgency": "high", "impact": "single_user", "requested_action": "Fix the user's microphone in Teams", "routing_queue": "end_user_computing"}


**Why this cell — TODO 2, the comparison call:** the same ticket, the same two messages, the same temperature and cap — through the vendor API. The point is not the score. It is that **the only thing that changes is the endpoint name**, and everything else about the request is identical: that is what makes a comparison honest, and it is the same one-line switch OQ's integration would flip between a hosted pilot and the on-premises VM. Notice too what *else* changed: the ticket text has now left the machine.

If the key cell near the top found no `OPENAI_API_KEY`, this cell stops and says so; everything above it already ran.

In [13]:
assert key_ok, "No OPENAI_API_KEY - see the key cell near the top. Everything above this cell ran without it."

# ── TODO 2 ─────────────────────────────────────────────────────────
# Send ticket_messages through the vendor API: the SAME messages, temperature and cap as the local
# call above. Only the endpoint name changes.
# Hint: get_endpoint() knows "local", "hosted" and "tuned"; .chat(messages=..., temperature=..., max_tokens=...)
# returns the reply text.
hosted_llm = get_endpoint("hosted")
call_started = time.time()
hosted_reply = hosted_llm.chat(messages=ticket_messages, temperature=0.0, max_tokens=300, timeout=600)
# ───────────────────────────────────────────────────────────────────

assert hosted_llm is not ... and hosted_reply is not ..., "TODO 2 is not filled in yet"

hosted_result = {"ticket_id": ticket_pair["ticket_id"], "model": hosted_llm.model, "endpoint": repr(hosted_llm),
                 "reply": hosted_reply, "seconds": round(time.time() - call_started, 2)}
utils.save_json(CHECKPOINT_DIR, "02_ticket_hosted", hosted_result)

hosted_record, hosted_format = inference_utils.parse_json_reply(hosted_result["reply"])
hosted_marks = inference_utils.compare_records(hosted_record, exact_fields)

print(f"--- {hosted_llm.model} (vendor API) replied in {hosted_result['seconds']} s ---")
print(hosted_result["reply"])
print()
print(f"format : {hosted_format}")
print(f"fields : {inference_utils.marks_text(hosted_marks, len(exact_fields))}")

checkpoint saved: C:\Users\utkar\oq-advanced-ai\checkpoints\local\02_ticket_hosted.json
--- gpt-4o-mini (vendor API) replied in 2.44 s ---
{
  "category": "software",
  "affected_system": "TEAMS",
  "asset_tag": null,
  "urgency": "high",
  "impact": "single_user",
  "requested_action": "Resolve microphone issue in TEAMS",
  "routing_queue": "apps_support"
}

format : valid JSON object
fields : 4/6 right; wrong: affected_system, routing_queue


**Why this cell — milestone 4, one table:** the two replies side by side, with the rows that matter for the build-buy-host decision you drafted yesterday, not just the score. *Where it ran* and *who saw the ticket* are the residency rows — the gates in your decision matrix. *Seconds* is latency for **one** request, on this hardware, and says nothing yet about a queue of them (S8 does). *Format* and *fields* are the quality rows, on **one ticket** — a sample of one proves nothing; S12 scores twenty and still warns you about the sample size. The table is saved next to the other checkpoints, so it can be quoted later without re-running either model.

In [14]:
comparison_rows = [
    ("model",               local_result["model"],                    hosted_result["model"]),
    ("runs at",             local_llm.base_url,                       hosted_llm.base_url),
    ("who saw the ticket",  "this machine only",                      "the vendor's servers"),
    ("seconds, 1 request",  local_result["seconds"],                  hosted_result["seconds"]),
    ("format",              local_format,                             hosted_format),
    ("exact fields",        inference_utils.marks_text(local_marks, len(exact_fields)), inference_utils.marks_text(hosted_marks, len(exact_fields))),
    ("requested_action",    (local_record or {}).get("requested_action", "-"),
                            (hosted_record or {}).get("requested_action", "-")),
    ("expected action",     expected_record["requested_action"],      expected_record["requested_action"]),
]

comparison_table = inference_utils.side_by_side(comparison_rows, "self-hosted (Ollama)", "vendor API")
print(comparison_table)

comparison = {"ticket_id": ticket_pair["ticket_id"], "rows": comparison_rows, "table": comparison_table,
              "local": local_result, "hosted": hosted_result, "expected": expected_record}
saved_path = utils.save_json(CHECKPOINT_DIR, "02_comparison", comparison)

                       | self-hosted (Ollama)                 | vendor API                          
----------------------------------------------------------------------------------------------------
model                  | llama3.2:1b                          | gpt-4o-mini                         
runs at                | http://localhost:11435/v1            | https://api.openai.com/v1           
who saw the ticket     | this machine only                    | the vendor's servers                
seconds, 1 request     | 5.17                                 | 2.44                                
format                 | not JSON                             | valid JSON object                   
exact fields           | no readable record (0/6)             | 4/6 right; wrong: affected_system...
requested_action       | -                                    | Resolve microphone issue in TEAMS   
expected action        | Fix the user's microphone in Teams   | Fix the user's microphone i

**Why this cell:** the declared result in one block, so "done" can be checked from across the room, with the pull time kept apart from the run time. Read the last three lines as the hand-off to the next two sessions: the tokens-per-second figure is what S8 loads up until it breaks, and the fields the small model got wrong are what S11's fine-tune is for.

In [15]:
sitting_minutes = (time.time() - sitting_started) / 60

print("LOCAL INFERENCE READY")
print(f"  server        : Ollama {server['version']} at {server['url']}"
      f" ({'installed and ' if server['installed_here'] else ''}{'started by this notebook' if server['started_here'] else 'already running'})")
print(f"  model         : {card['name']} - {card['parameter_size']} parameters, {card['quantization']}, {card['disk_gb']} GB on disk")
print(f"  pull          : {pull_record['status']} - {pull_record['seconds']} s on {pull_record['when']} ({pull_record['where']})")
print(f"  hardware      : {gpu or 'no NVIDIA GPU'}; {warm['gpu_share']:.0%} of the model in GPU memory"
      f"{' (an integrated GPU, Ollama found one)' if gpu is None and warm['gpu_share'] > 0 else ''}")
print(f"  speed         : {speed['tokens_per_second']} tokens per second, one request at a time  -> S8 starts here")
print(f"  parameters    : " + ", ".join(f"{name} {'identical' if result['identical'] else 'different'}"
                                       for name, result in experiment_results.items()))
print(f"  ticket        : {ticket_pair['ticket_id']} - self-hosted {inference_utils.marks_text(local_marks, len(exact_fields))}"
      f" | vendor {inference_utils.marks_text(hosted_marks, len(exact_fields))}  -> S11 fixes the local column")
print(f"  saved         : {CHECKPOINT_DIR / '02_*.json'}")
print(f"  this sitting  : {sitting_minutes:.1f} minutes from the settings cell to here (pull included)")

LOCAL INFERENCE READY
  server        : Ollama 0.12.10 at http://localhost:11435 (already running)
  model         : llama3.2:1b - 1.2B parameters, Q8_0, 1.32 GB on disk
  pull          : already there - 2.1 s on 2026-09-22T19:20:55 (local)
  hardware      : no NVIDIA GPU; 0% of the model in GPU memory
  speed         : 37.3 tokens per second, one request at a time  -> S8 starts here
  parameters    : repeatable identical, creative different, capped identical
  ticket        : INC-004183 - self-hosted no readable record (0/6) | vendor 4/6 right; wrong: affected_system, routing_queue  -> S11 fixes the local column
  saved         : C:\Users\utkar\oq-advanced-ai\checkpoints\local\02_*.json
  this sitting  : 1.0 minutes from the settings cell to here (pull included)
